In [ ]:
#   import Library

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Masukkan model-model yang ingin kamu benchmark
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
# Load Data & Eksplirasi (Code Cell)

# Load dataset
df = pd.read_csv("UCI_Credit_Card.csv")

# Tampilkan informasi dasar data
print(f"Bentuk data: {df.shape}")
display(df.head())

# Tampilkan statistik deskriptif
display(df.describe())

In [ ]:
# Visualisasi Kolerasi Prediktor (Code Cell)
# Analisis Korelasi dengan Target
matrix_korelasi = df.corr()['default.payment.next.month'].drop(['ID', 'default.payment.next.month']).reset_index()
matrix_korelasi.columns = ['Nama Fitur', 'Nilai Korelasi']
matrix_korelasi['Korelasi Absolut'] = matrix_korelasi['Nilai Korelasi'].abs()
matrix_korelasi = matrix_korelasi.sort_values(by='Korelasi Absolut', ascending=False).reset_index(drop=True)

# Plot Grafik
plt.figure(figsize=(10, 5))
sns.barplot(x='Nilai Korelasi', y='Nama Fitur', data=matrix_korelasi.head(10), palette='coolwarm')
plt.title("Top 10 Fitur Prediktor Terkuat")
plt.show()

In [ ]:
# Define fiture & Pipeline Pra-Pemrosesan 

# Pisahkan X dan y
X = df.drop(columns=['ID', 'default.payment.next.month'])
y = df['default.payment.next.month']

# Tentukan kolom
kolom_numerik_kontinu = [
    'LIMIT_BAL', 'AGE',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]
kolom_kategorikal_status = [
    'SEX', 'EDUCATION', 'MARRIAGE',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'
]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=10, stratify=y
)

# Setup Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num_scale', StandardScaler(), kolom_numerik_kontinu),
        ('cat_keep', 'passthrough', kolom_kategorikal_status)
    ]
)
print("Pipeline preprocessor berhasil didefinisikan.")

In [ ]:
# Proses Benchmarking

all_models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', solver='liblinear', random_state=10),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=10),
    "Gradient Boosting": GradientBoostingClassifier(random_state=10),
    "XGBoost": XGBClassifier(random_state=10, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(random_state=10, is_unbalance=True, verbose=-1)
}

results = []
trained_pipelines = {}

for name, model_obj in all_models.items():
    print(f"Melatih model: {name}...")
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model_obj)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    # Ambil probabilitas untuk ROC-AUC
    if hasattr(pipeline.named_steps['classifier'], "predict_proba"):
        y_scores = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_scores)
    else:
        roc_auc = np.nan
        
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc
    })
    trained_pipelines[name] = pipeline

# Tampilkan Leaderboard
results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
# Menyimpan Model Terbaik

# Ambil model pemenang pertama
pemenang = results_df.iloc[0]['Model']
best_pipeline = trained_pipelines[pemenang]

# Simpan ke file pickle
os.makedirs("saved_models", exist_ok=True)
best_pkl_path = f"saved_models/BEST_{pemenang.replace(' ', '_')}.pkl"

with open(best_pkl_path, 'wb') as f:
    pickle.dump(best_pipeline, f)

print(f"Model terbaik ({pemenang}) berhasil disimpan di: {best_pkl_path}")